# 01 - Neo4j Basics & First Knowledge Graph

**Goal**: Learn Cypher query language and build your first knowledge graph in Neo4j.

**Prerequisites**: 
- Neo4j running via `docker compose up -d` from the project root
- `pip install neo4j python-dotenv`

**Time**: ~30 minutes

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
PASSWORD = os.getenv("NEO4J_PASSWORD", "graphrag-password")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()
print("Connected to Neo4j!")

## Step 1: Create Nodes and Relationships

Let's build a small knowledge graph about ML papers and authors.

In [ ]:
def run_query(query, parameters=None):
    """Helper to run a Cypher query and return results."""
    with driver.session() as session:
        result = session.run(query, parameters or {})
        return [record.data() for record in result]

# Clear existing data (careful in production!)
run_query("MATCH (n) DETACH DELETE n")
print("Database cleared.")

In [ ]:
# Create some ML papers and authors
setup_query = """
// Authors
CREATE (vaswani:Author {name: 'Ashish Vaswani'})
CREATE (hinton:Author {name: 'Geoffrey Hinton'})
CREATE (bengio:Author {name: 'Yoshua Bengio'})
CREATE (lecun:Author {name: 'Yann LeCun'})
CREATE (goodfellow:Author {name: 'Ian Goodfellow'})

// Papers
CREATE (attention:Paper {title: 'Attention Is All You Need', year: 2017})
CREATE (gan:Paper {title: 'Generative Adversarial Networks', year: 2014})
CREATE (dropout:Paper {title: 'Dropout: A Simple Way to Prevent Overfitting', year: 2014})
CREATE (deeplearning:Paper {title: 'Deep Learning (Nature)', year: 2015})

// Topics
CREATE (transformers:Topic {name: 'Transformers'})
CREATE (generative:Topic {name: 'Generative Models'})
CREATE (regularization:Topic {name: 'Regularization'})
CREATE (nlp:Topic {name: 'NLP'})
CREATE (cv:Topic {name: 'Computer Vision'})

// Relationships: Author -[:WROTE]-> Paper
CREATE (vaswani)-[:WROTE]->(attention)
CREATE (goodfellow)-[:WROTE]->(gan)
CREATE (hinton)-[:WROTE]->(dropout)
CREATE (hinton)-[:WROTE]->(deeplearning)
CREATE (bengio)-[:WROTE]->(deeplearning)
CREATE (lecun)-[:WROTE]->(deeplearning)

// Relationships: Paper -[:ABOUT]-> Topic
CREATE (attention)-[:ABOUT]->(transformers)
CREATE (attention)-[:ABOUT]->(nlp)
CREATE (gan)-[:ABOUT]->(generative)
CREATE (gan)-[:ABOUT]->(cv)
CREATE (dropout)-[:ABOUT]->(regularization)
CREATE (deeplearning)-[:ABOUT]->(nlp)
CREATE (deeplearning)-[:ABOUT]->(cv)

// Relationships: Paper -[:CITES]-> Paper
CREATE (attention)-[:CITES]->(dropout)
CREATE (gan)-[:CITES]->(dropout)

RETURN 'Knowledge graph created!' AS status
"""

result = run_query(setup_query)
print(result)

## Step 2: Query the Graph

Now let's explore with Cypher queries — this is where the graph structure shines.

In [ ]:
# Simple: Find all papers by Hinton
results = run_query("""
    MATCH (a:Author {name: 'Geoffrey Hinton'})-[:WROTE]->(p:Paper)
    RETURN a.name AS author, p.title AS paper, p.year AS year
""")
for r in results:
    print(f"  {r['author']} wrote '{r['paper']}' ({r['year']})")

In [ ]:
# Multi-hop: Which authors wrote about topics that Hinton also wrote about?
results = run_query("""
    MATCH (hinton:Author {name: 'Geoffrey Hinton'})-[:WROTE]->(p1:Paper)-[:ABOUT]->(t:Topic)
          <-[:ABOUT]-(p2:Paper)<-[:WROTE]-(other:Author)
    WHERE other.name <> 'Geoffrey Hinton'
    RETURN DISTINCT other.name AS colleague, collect(DISTINCT t.name) AS shared_topics
""")
print("Authors who share topics with Hinton:")
for r in results:
    print(f"  {r['colleague']} — shared topics: {r['shared_topics']}")

In [ ]:
# Citation chain: What papers cite papers by Hinton?
results = run_query("""
    MATCH (a:Author {name: 'Geoffrey Hinton'})-[:WROTE]->(cited:Paper)
          <-[:CITES]-(citing:Paper)<-[:WROTE]-(citing_author:Author)
    RETURN citing_author.name AS who_cited,
           citing.title AS their_paper,
           cited.title AS hinton_paper
""")
print("Papers that cite Hinton's work:")
for r in results:
    print(f"  '{r['their_paper']}' by {r['who_cited']} cites '{r['hinton_paper']}'")

## Step 3: Visualize the Graph

Use pyvis to create an interactive HTML visualization.

In [ ]:
from pyvis.network import Network

# Fetch all nodes and relationships
nodes_data = run_query("""
    MATCH (n)
    RETURN id(n) AS id, labels(n)[0] AS label, 
           coalesce(n.name, n.title) AS name
""")

edges_data = run_query("""
    MATCH (a)-[r]->(b)
    RETURN id(a) AS source, id(b) AS target, type(r) AS rel_type
""")

# Build visualization
net = Network(notebook=True, height="600px", width="100%", cdn_resources="remote")

colors = {"Author": "#4CAF50", "Paper": "#2196F3", "Topic": "#FF9800"}

for node in nodes_data:
    net.add_node(
        node["id"], 
        label=node["name"], 
        color=colors.get(node["label"], "#gray"),
        title=f"{node['label']}: {node['name']}"
    )

for edge in edges_data:
    net.add_edge(edge["source"], edge["target"], label=edge["rel_type"])

net.show("../data/graph_viz.html")
print("Graph saved to data/graph_viz.html — open in browser!")

## Next Steps

Now that you understand nodes, relationships, and Cypher queries, move on to:
- **02_graphrag_quickstart.ipynb** — Run Microsoft GraphRAG on sample text
- **03_build_your_project.ipynb** — Build the full pipeline with your chosen dataset

### Key takeaways from this notebook:
1. **Nodes** = entities (Author, Paper, Topic)
2. **Relationships** = connections (WROTE, ABOUT, CITES)
3. **Cypher** = SQL for graphs — `MATCH` patterns, `WHERE` filters, `RETURN` results
4. **Multi-hop queries** are where graphs beat tables — follow chains of relationships naturally

In [ ]:
# Cleanup
driver.close()
print("Done!")